<a href="https://colab.research.google.com/github/Aswathi281099/Generative-Artificial-Intelligence/blob/main/AI_TASK_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
from collections import defaultdict
import numpy as np

In [2]:
class CustomBPETokenizer:
    """
    Byte-Pair Encoding (BPE) Tokenizer constructed from scratch.
    Handles vocabulary creation via iterative merging and bidirectional text conversion.
    """
    def __init__(self, vocab_size: int = 256):
        self.target_vocab_size = vocab_size
        self.pad_token = "<PAD>"
        self.unk_token = "<UNK>"
        self.bos_token = "<BOS>"
        self.eos_token = "<EOS>"

        self.vocab: dict[str, int] = {}
        self.inverse_vocab: dict[int, str] = {}
        self.merges: dict[tuple[str, str], str] = {}

    def _get_stats(self, corpus_words: dict[tuple[str, ...], int]) -> dict[tuple[str, str], int]:
        """Calculates frequencies of adjacent symbol pairs across the corpus."""
        pairs = defaultdict(int)
        for word, freq in corpus_words.items():
            for i in range(len(word) - 1):
                pairs[(word[i], word[i+1])] += freq
        return pairs

    def _merge_corpus(
        self,
        pair: tuple[str, str],
        corpus_words: dict[tuple[str, ...], int]
    ) -> dict[tuple[str, ...], int]:
        """Replaces instances of `pair` with their merged representation."""
        bigram = re.escape(' '.join(pair))
        pattern = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')

        new_corpus = {}
        for word, freq in corpus_words.items():
            word_str = ' '.join(word)
            new_word_str = pattern.sub(''.join(pair), word_str)
            new_word = tuple(new_word_str.split())
            new_corpus[new_word] = freq
        return new_corpus

    def fit(self, text: str):
        """Builds BPE vocabulary and merge rules up to `target_vocab_size`."""
        words = text.strip().split()
        corpus_words = defaultdict(int)
        for word in words:
            # Append </w> as an end-of-word boundary indicator
            char_tuple = tuple(list(word) + ['</w>'])
            corpus_words[char_tuple] += 1

        # Initialize vocab with base special tokens and individual unique characters
        unique_chars = sorted(list(set(c for word in corpus_words for c in word)))
        base_tokens = [self.pad_token, self.unk_token, self.bos_token, self.eos_token] + unique_chars

        self.vocab = {token: idx for idx, token in enumerate(base_tokens)}
        self.inverse_vocab = {idx: token for token, idx in self.vocab.items()}

        num_merges = self.target_vocab_size - len(self.vocab)

        for _ in range(num_merges):
            pairs = self._get_stats(corpus_words)
            if not pairs:
                break

            # Identify most frequent adjacent pair
            best_pair = max(pairs, key=pairs.get)
            merged_token = ''.join(best_pair)

            # Register new token and merge rule
            new_id = len(self.vocab)
            self.vocab[merged_token] = new_id
            self.inverse_vocab[new_id] = merged_token
            self.merges[best_pair] = merged_token

            # Apply merge across corpus
            corpus_words = self._merge_corpus(best_pair, corpus_words)

    def encode(self, text: str) -> list[int]:
        """Converts raw text into a list of subword token IDs."""
        words = text.strip().split()
        encoded_ids = []

        for word in words:
            tokens = list(word) + ['</w>']

            while len(tokens) >= 2:
                pairs = [(tokens[i], tokens[i+1]) for i in range(len(tokens) - 1)]
                mergeable_pairs = [p for p in pairs if p in self.merges]
                if not mergeable_pairs:
                    break

                # Apply highest priority (earliest learned) merge rule
                best_pair = min(mergeable_pairs, key=lambda p: list(self.merges.keys()).index(p))

                i = 0
                new_tokens = []
                while i < len(tokens):
                    if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == best_pair:
                        new_tokens.append(self.merges[best_pair])
                        i += 2
                    else:
                        new_tokens.append(tokens[i])
                        i += 1
                tokens = new_tokens

            for tok in tokens:
                encoded_ids.append(self.vocab.get(tok, self.vocab[self.unk_token]))

        return encoded_ids

    def decode(self, token_ids: list[int]) -> str:
        """Converts subword token IDs back to a text string."""
        tokens = [self.inverse_vocab.get(idx, self.unk_token) for idx in token_ids]
        raw_text = "".join(tokens)
        return raw_text.replace("</w>", " ").strip()

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CausalSelfAttention(nn.Module):
    """Multi-Head Causal Attention using custom lower-triangular masking."""
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, S, E = x.shape  # (Batch, Sequence Length, Embedding Dim)

        # Linear projections into 4D Multi-Head shape: (B, H, S, d_k)
        Q = self.q_proj(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        K = self.k_proj(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        V = self.v_proj(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)

        # Raw attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / (self.d_k ** 0.5)

        # Construct Causal Masking Matrix (Upper triangle above diagonal set to True)
        causal_mask = torch.triu(torch.ones((S, S), device=x.device, dtype=torch.bool), diagonal=1)

        # Apply mask: fill future positions with -inf so softmax evaluates them to 0
        scores = scores.masked_fill(causal_mask, float('-inf'))

        # Softmax & Value aggregation
        attn_weights = F.softmax(scores, dim=-1)
        context = torch.matmul(attn_weights, V)  # (B, H, S, d_k)

        # Re-assemble head dimensions: (B, S, d_model)
        context = context.transpose(1, 2).contiguous().view(B, S, E)
        return self.out_proj(context)


class TransformerBlock(nn.Module):
    """Transformer Decoder Block with Self-Attention and Feed-Forward Network."""
    def __init__(self, d_model: int, num_heads: int, d_ff: int):
        super().__init__()
        self.attn = CausalSelfAttention(d_model, num_heads)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class AutoregressiveCausalLM(nn.Module):
    """Decoder-only Autoregressive Language Model."""
    def __init__(self, vocab_size: int, d_model: int = 128, num_heads: int = 4, num_layers: int = 2, max_seq_len: int = 64):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff=d_model * 4)
            for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, S = idx.shape
        positions = torch.arange(0, S, device=idx.device).unsqueeze(0)

        x = self.token_emb(idx) + self.pos_emb(positions)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.head(x)  # (B, S, vocab_size)

In [4]:
# 1. Corpus Preparation
corpus = (
    "scaled dot product attention computes alignment scores between queries and keys. "
    "causal language models predict future tokens given past context. "
    "byte pair encoding iteratively merges frequent character pairs into subwords. "
    "autoregressive sequence generation predicts one token at a time."
)

# 2. Fit Tokenizer
tokenizer = CustomBPETokenizer(vocab_size=75)
tokenizer.fit(corpus)
encoded_corpus = tokenizer.encode(corpus)

print(f"Vocabulary Size: {len(tokenizer.vocab)}")
print(f"Total Tokens: {len(encoded_corpus)}")

# 3. Create Causal Shifted Inputs (X) and Targets (Y)
seq_len = 16
X_list, Y_list = [], []
for i in range(0, len(encoded_corpus) - seq_len):
    X_list.append(encoded_corpus[i : i + seq_len])
    Y_list.append(encoded_corpus[i + 1 : i + seq_len + 1])  # Target shifted by 1 token

X_tensor = torch.tensor(X_list, dtype=torch.long)
Y_tensor = torch.tensor(Y_list, dtype=torch.long)

# 4. Initialize Model, Optimizer, and Criterion
model = AutoregressiveCausalLM(
    vocab_size=len(tokenizer.vocab),
    d_model=64,
    num_heads=4,
    num_layers=2,
    max_seq_len=seq_len
)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3)
criterion = nn.CrossEntropyLoss()

# 5. Training Loop
model.train()
epochs = 200
print("\n--- Training Progress ---")
for epoch in range(1, epochs + 1):
    optimizer.zero_grad()

    # Forward Pass
    logits = model(X_tensor)

    # Reshape logits to (Batch * Seq_Len, Vocab_Size) for CrossEntropy Loss
    loss = criterion(logits.view(-1, logits.size(-1)), Y_tensor.view(-1))

    loss.backward()
    optimizer.step()

    if epoch % 40 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss.item():.4f}")

# 6. Autoregressive Greedy Text Generation
def generate_text(model, tokenizer, prompt: str, max_new_tokens: int = 12) -> str:
    model.eval()
    tokens = tokenizer.encode(prompt)
    input_tensor = torch.tensor([tokens], dtype=torch.long)

    for _ in range(max_new_tokens):
        # Limit context size to max_seq_len
        cond_tensor = input_tensor[:, -seq_len:]
        with torch.no_grad():
            logits = model(cond_tensor)

        # Select highest probability token from the final position
        next_token_logits = logits[:, -1, :]
        next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)

        input_tensor = torch.cat((input_tensor, next_token), dim=1)

    generated_ids = input_tensor[0].tolist()
    return tokenizer.decode(generated_ids)

# Execution Test
prompt = "causal language models"
generated_output = generate_text(model, tokenizer, prompt, max_new_tokens=10)

print("\n--- Generation Test ---")
print(f"Input Prompt: '{prompt}'")
print(f"Generated:    '{generated_output}'")

Vocabulary Size: 75
Total Tokens: 164

--- Training Progress ---
Epoch 040 | Loss: 0.7822
Epoch 080 | Loss: 0.1027
Epoch 120 | Loss: 0.0816
Epoch 160 | Loss: 0.0769
Epoch 200 | Loss: 0.0749

--- Generation Test ---
Input Prompt: 'causal language models'
Generated:    'causal language models predict future token'
